# Number-plate detection, PaddleOCR and smoke predictions
Run cells in order in Google Colab. Upload images or a ZIP when prompted.

Changes: preserve raw OCR characters; process every detected plate in every image; use a configurable smoke threshold of 0.40; save independent ANPR and smoke results plus JSON. No colour-based smoke detector or invented severity/confidence percentages.

Smoke boxes are predictions, not confirmed smoke or evidence that a vehicle is emitting smoke. Box coverage is not smoke-pixel area. A higher threshold can miss faint smoke. Validate on labelled smoke and non-smoke examples before use. OCR scores and detector scores are not measured accuracy.

This notebook was syntax-checked and its post-processing tested; model inference has not been rerun by the author of this revision.

In [ ]:
# 1. Install (if Colab asks for a runtime restart, restart then continue below)
%pip -q install ultralytics huggingface_hub paddlepaddle paddleocr matplotlib opencv-python

In [ ]:
# 2. Imports and settings
import cv2
import json
import re
import shutil
import zipfile
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone
from google.colab import files
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
from paddleocr import PaddleOCR

PLATE_CONFIDENCE = 0.25
SMOKE_CONFIDENCE = 0.40  # starting value; validate on your own images
ENABLE_SMOKE = True
IMAGE_SIZE = 1280
# Each run gets its own folder so old inputs are not processed accidentally.
run_directory = Path('anpr_runs') / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
input_directory = run_directory / 'inputs'
output_directory = run_directory / 'outputs'
input_directory.mkdir(parents=True)
output_directory.mkdir()
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

In [ ]:
# 3. Load models
plate_path = hf_hub_download(
    repo_id='Koushim/yolov8-license-plate-detection',
    filename='best.pt', token=False,
)
plate_model = YOLO(plate_path)
print('Plate classes:', plate_model.names)
ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    enable_mkldnn=False,
)
smoke_model = None
smoke_load_error = None
if ENABLE_SMOKE:
    try:
        smoke_path = hf_hub_download(
            repo_id='nobita111222/fire-smoke-yolov8n',
            filename='best.pt', token=False,
        )
        smoke_model = YOLO(smoke_path)
        print('Smoke model classes:', smoke_model.names)
        if not any(str(v).lower() == 'smoke' for v in smoke_model.names.values()):
            raise ValueError('This checkpoint has no smoke class.')
    except Exception as exc:
        smoke_model = None
        smoke_load_error = str(exc)
        print('Smoke model unavailable; ANPR can continue:', exc)
print('ANPR models ready.')

In [ ]:
# 4. Detection helpers

def deduplicate_boxes(detections):
    if not detections:
        return []
    boxes = [[x1, y1, x2-x1, y2-y1] for x1,y1,x2,y2,_ in detections]
    scores = [d[4] for d in detections]
    indices = cv2.dnn.NMSBoxes(boxes, scores, 0.0, 0.45)
    return [detections[int(i)] for i in np.asarray(indices).reshape(-1)]

def clip_box(box, width, height):
    x1,y1,x2,y2 = map(int,box)
    return (max(0,min(width,x1)), max(0,min(height,y1)),
            max(0,min(width,x2)), max(0,min(height,y2)))

def detect_plate_boxes(image):
    """Detect number plates. Retry with overlapping crops if the full image fails."""
    full_results = plate_model.predict(
        source=image,
        imgsz=IMAGE_SIZE,
        conf=PLATE_CONFIDENCE,
        iou=0.45,
        max_det=10,
        verbose=False,
    )[0]

    detections = []
    if full_results.boxes is not None:
        for box in full_results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            confidence = float(box.conf[0])
            detections.append((x1, y1, x2, y2, confidence))

    if detections:
        return deduplicate_boxes(detections)

    # Small CCTV plates may disappear when the complete frame is resized.
    # Retry using four overlapping image tiles.
    height, width = image.shape[:2]
    tile_width = int(width * 0.65)
    tile_height = int(height * 0.65)
    starts = [
        (0, 0),
        (width - tile_width, 0),
        (0, height - tile_height),
        (width - tile_width, height - tile_height),
    ]

    for offset_x, offset_y in starts:
        tile = image[
            offset_y : offset_y + tile_height,
            offset_x : offset_x + tile_width,
        ]
        tile_result = plate_model.predict(
            source=tile,
            imgsz=IMAGE_SIZE,
            conf=PLATE_CONFIDENCE,
            iou=0.45,
            max_det=5,
            verbose=False,
        )[0]

        if tile_result.boxes is None:
            continue

        for box in tile_result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            confidence = float(box.conf[0])
            detections.append(
                (
                    x1 + offset_x,
                    y1 + offset_y,
                    x2 + offset_x,
                    y2 + offset_y,
                    confidence,
                )
            )

    return deduplicate_boxes(detections)



def draw_label(canvas, box, text, color):
    x1,y1,x2,y2 = box
    cv2.rectangle(canvas,(x1,y1),(x2,y2),color,2)
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 0.6
    (tw,th),baseline = cv2.getTextSize(text,font,scale,2)
    while tw > canvas.shape[1]-8 and scale > 0.2:
        scale *= 0.85
        (tw,th),baseline = cv2.getTextSize(text,font,scale,2)
    tx = max(3,min(x1,canvas.shape[1]-tw-4))
    ty = max(th+4,min(y1-8,canvas.shape[0]-baseline-4))
    cv2.rectangle(canvas,(tx-2,ty-th-3),(tx+tw+2,ty+baseline+2),(0,0,0),-1)
    cv2.putText(canvas,text,(tx,ty),font,scale,color,2,cv2.LINE_AA)

def save_image(path, image):
    if not cv2.imwrite(str(path), image):
        raise IOError(f'Could not save {path}')

In [ ]:
# 5. OCR helpers: preserve characters; never force 6 -> G or 1 -> I

def preprocess_plate(crop):
    upscaled = cv2.resize(crop,None,fx=5,fy=5,interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(upscaled,cv2.COLOR_BGR2GRAY)
    enhanced = cv2.createCLAHE(clipLimit=2.5,tileGridSize=(8,8)).apply(gray)
    denoised = cv2.bilateralFilter(enhanced,7,50,50)
    sharp = cv2.filter2D(denoised,-1,np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]))
    binary = cv2.adaptiveThreshold(sharp,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY,31,7)
    # PaddleOCR ndarray inputs use OpenCV BGR ordering.
    return {'upscaled':upscaled,
            'cleaned':cv2.cvtColor(sharp,cv2.COLOR_GRAY2BGR),
            'binary':cv2.cvtColor(binary,cv2.COLOR_GRAY2BGR)}

def run_ocr_variant(img):
    candidates = []
    for result in ocr.predict(img):
        data = result.json
        if callable(data):
            data = data()
        if isinstance(data,str):
            data = json.loads(data)
        data = data.get('res',data)
        texts = data.get('rec_texts',[])
        scores = data.get('rec_scores',[])
        polygons = data.get('rec_polys',[])
        pieces = []
        for i,(raw,score) in enumerate(zip(texts,scores)):
            if raw is None:
                continue
            text = re.sub(r'[^A-Z0-9]','',str(raw).upper())
            if not text or text == 'IND':
                continue
            if i < len(polygons):
                poly = np.asarray(polygons[i])
                cx,cy = poly.mean(axis=0)
                height = max(1.0,float(np.ptp(poly[:,1])))
            else:
                cx,cy,height = i,0,1
            pieces.append({'text':text,'raw':str(raw),'score':float(score),
                           'cx':float(cx),'cy':float(cy),'height':height})
        if not pieces:
            continue
        # Group approximately aligned text into rows, then read left to right.
        rows = []
        for p in sorted(pieces,key=lambda p:p['cy']):
            if rows and abs(p['cy']-np.mean([q['cy'] for q in rows[-1]])) <= 0.6*max(p['height'],np.mean([q['height'] for q in rows[-1]])):
                rows[-1].append(p)
            else:
                rows.append([p])
        ordered = [p for row in rows for p in sorted(row,key=lambda p:p['cx'])]
        candidates.append({'text':''.join(p['text'] for p in ordered),
            'raw_texts':[p['raw'] for p in ordered],
            'score':sum(p['score']*len(p['text']) for p in ordered)/sum(len(p['text']) for p in ordered)})
    return candidates

def choose_ocr(candidates):
    if not candidates:
        return None
    # Count agreement across independent preprocessing variants, not duplicate boxes.
    votes = {}
    for c in candidates:
        votes.setdefault(c['text'],set()).add(c['source'])
    best = dict(max(candidates,key=lambda c:(len(votes[c['text']]),c['score'])))
    best['variant_agreement'] = len(votes[best['text']])
    # A format hint only, not validation that a registration is real.
    best['common_plate_format'] = bool(re.fullmatch(r'[A-Z]{2}[0-9]{1,2}[A-Z]{1,3}[0-9]{4}',best['text']))
    return best

In [ ]:
# 6. Upload images or ZIP; image-only extraction avoids unsafe archive paths
uploaded = files.upload()
if not uploaded:
    raise ValueError('No files uploaded.')
for upload_index,(name,blob) in enumerate(uploaded.items()):
    basename = Path(name).name
    if Path(basename).suffix.lower() == '.zip':
        import io
        with zipfile.ZipFile(io.BytesIO(blob)) as archive:
            members = [m for m in archive.infolist()
                       if not m.is_dir() and Path(m.filename).suffix.lower() in IMAGE_EXTENSIONS]
            if sum(m.file_size for m in members) > 500*1024*1024:
                raise ValueError('ZIP images exceed the 500 MiB extraction limit.')
            for i,member in enumerate(members):
                # Flatten ZIP paths; preserve uniqueness with numeric prefixes.
                target = input_directory / f'{upload_index}_{i}_{Path(member.filename).name}'
                target.write_bytes(archive.read(member))
    elif Path(basename).suffix.lower() in IMAGE_EXTENSIONS:
        (input_directory / f'{upload_index}_{basename}').write_bytes(blob)
image_paths = sorted(p for p in input_directory.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
if not image_paths:
    raise ValueError('No supported images found.')
print('Images:',len(image_paths))

In [ ]:
# 7. Process every image and every plate; save separate ANPR and smoke outputs
all_results = []
for image_path in image_paths:
    image = cv2.imread(str(image_path))
    if image is None:
        all_results.append({'image':image_path.name,'error':'Cannot decode image'})
        continue
    height,width = image.shape[:2]
    key = image_path.stem + '_' + hashlib.sha256(image_path.name.encode()).hexdigest()[:8]
    anpr_output = image.copy()
    smoke_output = image.copy()
    record = {'image':image_path.name,'plates':[], 'smoke_predictions':[],
              'smoke_threshold':SMOKE_CONFIDENCE,'smoke_status':'disabled',
              'errors':[]}
    detections = detect_plate_boxes(image)
    for index,detection in enumerate(detections,1):
        box = clip_box(detection[:4],width,height)
        x1,y1,x2,y2 = box
        if x2<=x1 or y2<=y1:
            continue
        crop = image[y1:y2,x1:x2].copy()
        save_image(output_directory/f'{key}_plate_{index}.png',crop)
        candidates = []
        for source,variant in preprocess_plate(crop).items():
            try:
                for c in run_ocr_variant(variant):
                    c['source'] = source
                    candidates.append(c)
            except Exception as exc:
                record['errors'].append(f'Plate {index}, OCR {source}: {exc}')
        best = choose_ocr(candidates)
        plate_record = {'box':list(box),'detection_score':float(detection[4]),
                        'ocr':best,'ocr_candidates':candidates}
        record['plates'].append(plate_record)
        label = best['text'] if best else 'PLATE: OCR unreadable'
        draw_label(anpr_output,box,label,(0,255,0))
    if not record['plates']:
        cv2.putText(anpr_output,'No plate detected',(10,30),cv2.FONT_HERSHEY_SIMPLEX,
                    0.65,(0,0,255),2)
    if ENABLE_SMOKE and smoke_model is None:
        record['smoke_status'] = 'model_unavailable'
        record['errors'].append(smoke_load_error or 'Smoke model unavailable')
    elif ENABLE_SMOKE:
        try:
            result = smoke_model.predict(source=image,conf=SMOKE_CONFIDENCE,
                       iou=0.45,imgsz=IMAGE_SIZE,max_det=20,verbose=False)[0]
            coverage = np.zeros((height,width),dtype=np.uint8)
            if result.boxes is not None:
                for b in result.boxes:
                    if str(smoke_model.names[int(b.cls[0])]).lower() != 'smoke':
                        continue
                    box = clip_box(b.xyxy[0].tolist(),width,height)
                    x1,y1,x2,y2 = box
                    if x2<=x1 or y2<=y1:
                        continue
                    score = float(b.conf[0])
                    record['smoke_predictions'].append({'box':list(box),'model_score':score})
                    coverage[y1:y2,x1:x2] = 1
                    draw_label(smoke_output,box,f'Smoke prediction {score:.2f}',(0,0,255))
            record['smoke_status'] = ('predictions_above_threshold' if record['smoke_predictions']
                                      else 'no_predictions_above_threshold')
            record['smoke_box_coverage_percent'] = float(coverage.mean()*100)
            record['coverage_note'] = 'Union of bounding boxes; NOT actual smoke-pixel area or severity.'
        except Exception as exc:
            record['smoke_status'] = 'inference_error'
            record['errors'].append(str(exc))
    if not record['smoke_predictions']:
        message = ('No smoke prediction above threshold' if record['smoke_status']=='no_predictions_above_threshold'
                   else 'Smoke: '+record['smoke_status'])
        cv2.putText(smoke_output,message,(10,30),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,200,255),2)
    save_image(output_directory/f'{key}_ANPR.jpg',anpr_output)
    if ENABLE_SMOKE:
        save_image(output_directory/f'{key}_smoke.jpg',smoke_output)
    (output_directory/f'{key}_results.json').write_text(json.dumps(record,indent=2))
    all_results.append(record)
    print('Image:',image_path.name)
    for i,p in enumerate(record['plates'],1):
        best = p['ocr']
        print(f"Plate {i}: {best['text'] if best else 'unreadable'} | detector score {p['detection_score']:.3f}")
        if best:
            print(f"  OCR score {best['score']:.3f}; agreement {best['variant_agreement']}/3 variants")
    print('Smoke:',record['smoke_status'])
    for p in record['smoke_predictions']:
        print('  Prediction score:',round(p['model_score'],3),'Box:',p['box'])
    for error in record['errors']:
        print('Issue:',error)
    fig,axes = plt.subplots(1,2 if ENABLE_SMOKE else 1,figsize=(16,7))
    axes = np.atleast_1d(axes)
    axes[0].imshow(cv2.cvtColor(anpr_output,cv2.COLOR_BGR2RGB))
    axes[0].set_title('Number plates and raw OCR')
    if ENABLE_SMOKE:
        axes[1].imshow(cv2.cvtColor(smoke_output,cv2.COLOR_BGR2RGB))
        axes[1].set_title(f'Smoke predictions; threshold {SMOKE_CONFIDENCE:.2f}')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    plt.close(fig)
(output_directory/'all_results.json').write_text(json.dumps(all_results,indent=2))
print('Saved results:',output_directory)

In [ ]:
# 8. Download outputs (annotated images, plate crops and JSON)
archive_path = shutil.make_archive(str(run_directory/'ANPR_smoke_results'),'zip',output_directory)
files.download(archive_path)